# 🌦️ GuardiánClima ITBA
### Challenge Tecnológico Integrador

**Instrucciones:**
1. Ejecutá la **Celda 1** (ID: `ectRF4AvH_uk`) para instalar las librerías principales.
2. **Configurá tus API Keys en Colab Secrets** (instrucciones en la celda siguiente) con los nombres `OPENWEATHERMAP_API_KEY` y `GEMINI_API_KEY`.
3. Ejecutá la **Celda 2** (ID: `db890def`) para cargar tus API Keys desde Colab Secrets.
4. Ejecutá la **Celda 3** (ID: `CdVNwLZHIKW1`) para configurar la IA de consejos de vestimenta.
5. Ejecutá la **Celda 4** (ID: `wZnBXwS8IJ5r`) para listar las versiones de IA.
6. Finalmente, ejecutá la **Celda 5** (ID: `wgYokAkRIJaw`) para lanzar la aplicación.
7. Para descargar los Archivos CSV generados, ejecutá la **Celda 6** (ID: `M2836VjOIVTd`).

> ⚠️ Si la sesión de Colab se reinicia, volvé a ejecutar desde la Celda 1.

In [ ]:
# ============================================================
# CELDA 1 — Instalación de librerías necesarias
# Ejecutá esta celda UNA SOLA VEZ al inicio de cada sesión.
# ============================================================
!pip install requests google-generativeai --quiet
print('✅ Librerías instaladas correctamente.')

## 🔑 Configuración de API Keys en Colab Secrets

Para que la aplicación funcione, necesitás configurar dos API Keys en el Administrador de Secretos de Colab:

1.  **OpenWeatherMap API Key**: Obtenela registrándote en [OpenWeatherMap](https://openweathermap.org/api).
2.  **Google Gemini API Key**: Obtenela en [Google AI Studio](https://makersuite.google.com/app/apikey).

**Pasos para configurar las claves:**

1.  Hacé clic en el **ícono de la llave** (🔑) en la barra lateral izquierda de Colab.
2.  Hacé clic en **"Agregar un secreto nuevo"**.
3.  Para la clave de OpenWeatherMap:
    *   En el campo "Nombre", escribí `OPENWEATHERMAP_API_KEY`.
    *   En el campo "Valor", pegá tu clave API de OpenWeatherMap.
    *   Asegurate de que la opción "Notebook access" esté activada para este secreto.
4.  Repetí el paso 3 para la clave de Google Gemini:
    *   En el campo "Nombre", escribí `GEMINI_API_KEY`.
    *   En el campo "Valor", pegá tu clave API de Google Gemini.
    *   Asegurate de que la opción "Notebook access" esté activada para este secreto.

Una vez que hayas configurado ambas claves, ejecutá la **Celda 2**.

In [ ]:
# ============================================================
# CELDA 2 — Configuración de API Keys y archivos
# Estas keys ahora se cargan desde el Administrador de Secretos de Colab.
# ============================================================
from google.colab import userdata

# --- API Keys ---
# Recupera las claves del Administrador de Secretos de Colab
API_KEY_OWM    = userdata.get("OPENWEATHERMAP_API_KEY")   # OpenWeatherMap
API_KEY_GEMINI = userdata.get("GEMINI_API_KEY")          # Google Gemini

# --- Nombres de archivos CSV ---
USUARIOS_CSV  = 'usuarios_simulados.csv'
HISTORIAL_CSV = 'historial_global.csv'

# Verificar si las claves se cargaron correctamente
if not API_KEY_OWM:
    print("❌ Advertencia: La API Key de OpenWeatherMap no se cargó. Asegúrate de haberla añadido como 'OPENWEATHERMAP_API_KEY' en Colab Secrets.")
if not API_KEY_GEMINI:
    print("❌ Advertencia: La API Key de Gemini no se cargó. Asegúrate de haberla añadido como 'GEMINI_API_KEY' en Colab Secrets.")

print('✅ Configuración cargada.')

In [ ]:
# ============================================================
# CELDA 3 — Configuración de la IA de consejos de vestimenta
# ============================================================
import google.generativeai as genai
def obtener_consejo_ia_gemini(api_key_gemini, temperatura, condicion_clima,
viento, humedad):
    """
    Obtiene un consejo de vestimenta usando la API de Gemini.
    """
    try:
        genai.configure(api_key=api_key_gemini)
        # Configuración del modelo (pueden explorar diferentes modelos si lo desean)
        # Para generación de texto simple, 'gemini-pro' suele ser una buena opción.
        model = genai.GenerativeModel('gemini-flash-latest')
        # Diseñar el prompt (este es un ejemplo, ustedes deben crear el suyo)
        # Es importante dar contexto y ser específico para obtener mejores resultados.
        prompt_diseñado_por_equipo = (
            f"Sos un asistente de moda personal. El clima actual es el siguiente:\n"
            f"- Temperatura: {temperatura} °C\n"
            f"- Condición: {condicion_clima}\n"
            f"- Viento: {viento} km/h\n"
            f"- Humedad: {humedad} %\n\n"
            f"Dame un consejo breve (máximo 5 oraciones) y práctico sobre "
            f"qué ropa usar hoy. Sé específico con prendas concretas."
        )
        print("\nGenerando consejo de vestimenta con IA...")
        # Generar contenido
        response = model.generate_content(prompt_diseñado_por_equipo)
        # Asegurarse de que hay texto en la respuesta
        if response.text:
            return response.text
        else:
            # A veces la API puede no devolver texto si hay problemas con el prompt o
            # filtros de seguridad
            # Investigar response.prompt_feedback si hay problemas
            print("La IA no pudo generar un consejo. Razón (si está disponible):",
            response.prompt_feedback)
            return "No se pudo generar un consejo en este momento."
    except Exception as e:
        print(f"Error al contactar la API de Gemini o procesar la respuesta: {e}")
        return "Error al generar el consejo de IA."

In [ ]:
# ============================================================
# CELDA 4 — Listado de versiones de IA
# ============================================================
import google.generativeai as genai

# Asegúrate de que API_KEY_GEMINI esté definido en un entorno global o pásalo como argumento
# En este caso, asumimos que API_KEY_GEMINI ya está definido en una celda anterior.

if 'API_KEY_GEMINI' in globals():
    try:
        genai.configure(api_key=API_KEY_GEMINI)
        print("Listando modelos disponibles para verificar el nombre correcto...")
        found_models = False
        for m in genai.list_models():
            if "generateContent" in m.supported_generation_methods:
                print(m.name)
                found_models = True
        if not found_models:
            print("No se encontraron modelos compatibles con 'generateContent'.")
    except Exception as e:
        print(f"Error al configurar la API de Gemini o listar modelos: {e}")
else:
    print("Error: La variable API_KEY_GEMINI no está definida. Asegúrate de ejecutar la celda de configuración de API Keys.")

In [ ]:
# ============================================================
# CELDA 5 — Lanzar aplicacion
# ============================================================
import csv
import os
import re
import requests
import google.generativeai as genai
from datetime import datetime
from collections import Counter


# ─────────────────────────────────────────
#  MÓDULO: Utilidades
# ─────────────────────────────────────────

def limpiar_pantalla():
    """Imprime líneas en blanco para simular limpieza de pantalla en Colab."""
    print('\n' * 3)


def inicializar_archivos():
    """Crea los archivos CSV con sus encabezados si todavía no existen."""
    if not os.path.exists(USUARIOS_CSV):
        with open(USUARIOS_CSV, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(['username', 'password_simulada'])
        print(f'Archivo {USUARIOS_CSV} creado.')

    if not os.path.exists(HISTORIAL_CSV):
        with open(HISTORIAL_CSV, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow([
                'NombreDeUsuario', 'Ciudad', 'FechaHora',
                'Temperatura_C', 'CondicionClima',
                'Humedad_Porcentaje', 'Viento_kmh'
            ])
        print(f'Archivo {HISTORIAL_CSV} creado.')


# ─────────────────────────────────────────
#  MÓDULO: Validación y gestión de usuarios
# ─────────────────────────────────────────

def validar_contrasena(password):
    """
    Verifica que la contraseña cumpla con los criterios de seguridad.
    Retorna una lista de errores (vacía si la contraseña es válida).
    Criterios:
      1. Mínimo 8 caracteres.
      2. Al menos una mayúscula.
      3. Al menos un número.
      4. Al menos un símbolo especial.
    """
    errores = []

    if len(password) < 8:
        errores.append('Debe tener al menos 8 caracteres.')
    if not re.search(r'[A-Z]', password):
        errores.append('Debe contener al menos una letra mayúscula (A-Z).')
    if not re.search(r'[0-9]', password):
        errores.append('Debe contener al menos un número (0-9).')
    if not re.search(r'[\W_]', password):
        errores.append('Debe contener al menos un símbolo (ej. !, @, #, $).')

    return errores


def registrar_nuevo_usuario():
    """Guía al usuario en el proceso de registro y devuelve el username o None."""
    limpiar_pantalla()
    print('─' * 40)
    print('     REGISTRO DE NUEVO USUARIO')
    print('─' * 40)

    # Elegir nombre de usuario único
    while True:
        username = input('Elegí un nombre de usuario: ').strip()
        if not username:
            print('⚠ El nombre de usuario no puede estar vacío.')
            continue

        try:
            with open(USUARIOS_CSV, 'r', newline='', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader)  # Saltar encabezado
                if any(row and row[0] == username for row in reader):
                    print('⚠ Ese nombre de usuario ya existe. Elegí otro.')
                    continue
        except FileNotFoundError:
            print('Error crítico: no se encontró el archivo de usuarios.')
            return None
        break

    # Elegir contraseña válida
    while True:
        password = input('Creá una contraseña: ')
        errores = validar_contrasena(password)

        if not errores:
            # Guardar nuevo usuario en el CSV
            with open(USUARIOS_CSV, 'a', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow([username, password])
            print(f'\n✅ ¡Registro exitoso! Bienvenido/a, {username}.')
            return username
        else:
            print('\n❌ La contraseña no cumple con los requisitos:')
            for e in errores:
                print(f'   - {e}')
            print('\n💡 Sugerencia: usá algo como "MiClima#2025" (mayúscula + número + símbolo).')

            seguir = input('¿Querés intentar con otra contraseña? (ingresá "s" para sí, cualquier otra tecla para cancelar el registro): ').lower()
            if seguir != 's':
                return None


def iniciar_sesion():
    """Verifica credenciales y devuelve el username si son correctas, o None."""
    limpiar_pantalla()
    print('─' * 40)
    print('           INICIAR SESIÓN')
    print('─' * 40)

    intentos = 3
    while intentos > 0:
        username = input('Nombre de usuario: ').strip()
        password = input('Contraseña: ')

        try:
            with open(USUARIOS_CSV, 'r', newline='', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader)  # Saltar encabezado
                for row in reader:
                    if row and len(row) > 1 and row[0] == username and row[1] == password:
                        print(f'\n✅ ¡Bienvenido/a de nuevo, {username}!')
                        return username
        except FileNotFoundError:
            print('Error crítico: no se encontró el archivo de usuarios.')
            return None

        intentos -= 1
        print(f'\n❌ Credenciales incorrectas. Te quedan {intentos} intento(s).')

    print('\n🔒 Agotaste los intentos. Volvé al menú de acceso.')
    return None


# ─────────────────────────────────────────
#  MÓDULO: Clima (OpenWeatherMap)
# ─────────────────────────────────────────

def obtener_clima(ciudad):
    """
    Consulta la API de OpenWeatherMap y retorna el JSON con los datos
    climáticos, o None si ocurre algún error.
    """
    url = 'https://api.openweathermap.org/data/2.5/weather'
    params = {
        'q': ciudad,
        'appid': API_KEY_OWM,
        'units': 'metric',
        'lang': 'es'
    }
    print(f'\nConsultando clima para "{ciudad}"...')
    try:
        respuesta = requests.get(url, params=params, timeout=10)
        respuesta.raise_for_status()
        return respuesta.json()
    except requests.exceptions.HTTPError:
        if respuesta.status_code == 401:
            print('❌ Error 401: API Key de OpenWeatherMap inválida.')
        elif respuesta.status_code == 404:
            print(f'❌ Error 404: Ciudad "{ciudad}" no encontrada.')
        else:
            print(f'❌ Error HTTP {respuesta.status_code}.')
        return None
    except requests.exceptions.RequestException as err:
        print(f'❌ Error de conexión: {err}')
        return None


def mostrar_clima(ciudad, datos):
    """Imprime los datos climáticos de forma clara."""
    try:
        temp      = datos['main']['temp']
        sensacion = datos['main']['feels_like']
        humedad   = datos['main']['humidity']
        condicion = datos['weather'][0]['description'].capitalize()
        viento_kmh = round(datos['wind']['speed'] * 3.6, 2)

        print(f'\n┌─ Clima en {ciudad.title()} ────────────────')
        print(f'│  🌡  Temperatura    : {temp} °C')
        print(f'│  🤔  Sensación      : {sensacion} °C')
        print(f'│  ⛅  Condición      : {condicion}')
        print(f'│  💧  Humedad        : {humedad} %')
        print(f'│  💨  Viento         : {viento_kmh} km/h')
        print('└──────────────────────────────────────')
    except (KeyError, TypeError):
        print('❌ Error al procesar los datos del clima.')


def guardar_en_historial(username, ciudad, datos):
    """Agrega una fila al historial global CSV con la consulta actual."""
    try:
        fecha_hora = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        temp       = datos['main']['temp']
        condicion  = datos['weather'][0]['description'].capitalize()
        humedad    = datos['main']['humidity']
        viento_kmh = round(datos['wind']['speed'] * 3.6, 2)

        with open(HISTORIAL_CSV, 'a', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow([
                username, ciudad.title(), fecha_hora,
                temp, condicion, humedad, viento_kmh
            ])
        print('✅ Consulta guardada en historial_global.csv')
        return True
    except (KeyError, TypeError):
        print('❌ Error al guardar en el historial.')
        return False


def consultar_clima_y_guardar(username):
    """Opción 1: pide ciudad, muestra clima y lo guarda. Devuelve los datos."""
    limpiar_pantalla()
    print('─' * 40)
    print('    CONSULTAR CLIMA ACTUAL')
    print('─' * 40)

    ciudad = input('Ingresá el nombre de una ciudad: ').strip()
    if not ciudad:
        print('⚠ El nombre de la ciudad no puede estar vacío.')
        return None

    datos = obtener_clima(ciudad)
    if datos:
        mostrar_clima(ciudad, datos)
        guardar_en_historial(username, ciudad, datos)
    return datos


def ver_historial_personal(username):
    """Opción 2: filtra y muestra el historial del usuario para una ciudad."""
    limpiar_pantalla()
    print('─' * 40)
    print('     MI HISTORIAL PERSONAL')
    print('─' * 40)

    ciudad = input('Ciudad a consultar en tu historial: ').strip().title()
    if not ciudad:
        print('⚠ El nombre de la ciudad no puede estar vacío.')
        return

    registros = []
    try:
        with open(HISTORIAL_CSV, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            encabezado = next(reader)
            for row in reader:
                if row and row[0] == username and row[1] == ciudad:
                    registros.append(dict(zip(encabezado, row)))
    except FileNotFoundError:
        print('❌ El archivo de historial no existe todavía.')
        return

    if registros:
        print(f'\n📋 Historial para "{ciudad}" ({len(registros)} registro(s)):')
        print(f'{"Fecha/Hora":<22} {"Temp (°C)":<12} {"Condición"}')
        print('─' * 55)
        for r in registros:
            print(f"{r['FechaHora']:<22} {r['Temperatura_C']:<12} {r['CondicionClima']}")
    else:
        print(f'\nNo hay registros tuyos para "{ciudad}".')


def mostrar_estadisticas_globales():
    """Opción 3: calcula y muestra estadísticas del historial completo."""
    limpiar_pantalla()
    print('─' * 40)
    print('   ESTADÍSTICAS GLOBALES DE USO')
    print('─' * 40)

    try:
        with open(HISTORIAL_CSV, 'r', newline='', encoding='utf-8') as f:
            filas = list(csv.reader(f))

        datos = filas[1:]  # Excluir encabezado
        if not datos:
            print('⚠ No hay consultas registradas todavía.')
            return

        ciudades      = [row[1] for row in datos if row]
        temperaturas  = [float(row[3]) for row in datos if row]
        ciudad_top    = Counter(ciudades).most_common(1)[0][0]
        total         = len(datos)
        temp_promedio = round(sum(temperaturas) / len(temperaturas), 2)

        print(f'\n🏆  Ciudad más consultada   : {ciudad_top}')
        print(f'📊  Total de consultas      : {total}')
        print(f'🌡   Temperatura promedio   : {temp_promedio} °C')
        print('\n💾  El archivo historial_global.csv está actualizado.')
        print('    Podés descargarlo desde el panel de archivos de Colab')
        print('    para generar gráficos en Excel (barras, líneas, torta).')

    except FileNotFoundError:
        print('❌ El archivo de historial no existe todavía.')
    except Exception as e:
        print(f'❌ Error al calcular estadísticas: {e}')


# ─────────────────────────────────────────
#  MÓDULO: IA (Google Gemini)
# ─────────────────────────────────────────

def obtener_consejo_ia(datos_clima):
    """
    Opción 4: envía los datos climáticos a Gemini y muestra el consejo
    de vestimenta generado.
    """
    limpiar_pantalla()
    print('─' * 40)
    print('   CONSEJO IA: ¿CÓMO ME VISTO HOY?')
    print('─' * 40)

    if not datos_clima:
        print('⚠ Primero realizá una consulta de clima (Opción 1).')
        return

    try:
        temp      = datos_clima['main']['temp']
        condicion = datos_clima['weather'][0]['description']
        viento    = round(datos_clima['wind']['speed'] * 3.6, 2)
        humedad   = datos_clima['main']['humidity']

        # Prompt diseñado para obtener consejos prácticos y concretos
        prompt = (
            f"Sos un asistente de moda personal. El clima actual es el siguiente:\n"
            f"- Temperatura: {temp} °C\n"
            f"- Condición: {condicion}\n"
            f"- Viento: {viento} km/h\n"
            f"- Humedad: {humedad} %\n\n"
            f"Dame un consejo breve (máximo 5 oraciones) y práctico sobre "
            f"qué ropa usar hoy. Sé específico con prendas concretas."
        )

        genai.configure(api_key=API_KEY_GEMINI)
        modelo = genai.GenerativeModel('gemini-flash-latest')

        print('\n⏳ Generando consejo con IA...')
        respuesta = modelo.generate_content(prompt)

        if respuesta.text:
            print('\n🤖 Consejo de GuardiánClima IA:')
            print('─' * 40)
            print(respuesta.text.strip())
            print('─' * 40)
        else:
            print('❌ La IA no pudo generar una respuesta en este momento.')

    except KeyError:
        print('❌ Error: datos de clima con formato inesperado.')
    except Exception as e:
        print(f'❌ Error al contactar la API de Gemini: {e}')


# ─────────────────────────────────────────
#  MÓDULO: Acerca de
# ─────────────────────────────────────────

def mostrar_acerca_de():
    """Opción 5: describe la aplicación, su funcionamiento y el equipo."""
    limpiar_pantalla()
    print('─' * 50)
    print('         ACERCA DE GUARDIÁNCLIMA ITBA')
    print('─' * 50)
    print()
    print('GuardiánClima ITBA es una aplicación de consola')
    print('desarrollada en Python como Challenge Tecnológico')
    print('Integrador de la materia Tecnología (ITBA).')
    print('(' * 50)
    print()
    print('── FUNCIONALIDADES ─────────────────────────────')
    print('1. Consulta de clima en tiempo real (OpenWeatherMap API)')
    print('   → Se guarda cada consulta en historial_global.csv')
    print('2. Historial personal filtrado por ciudad')
    print('3. Estadísticas globales de uso (ciudad top, promedio temp)')
    print('4. Consejo de vestimenta con IA generativa (Google Gemini)')
    print()
    print('── SEGURIDAD (ADVERTENCIA) ──────────────────────')
    print('⚠ SIMULACIÓN: Este sistema guarda contraseñas en texto')
    print('  plano en usuarios_simulados.csv. Esto es INSEGURO y')
    print('  solo se hace con fines educativos.')
    print('  En una app real, las contraseñas deben protegerse con')
    print('  técnicas de HASHING (ej. bcrypt, SHA-256 + salt) para')
    print('  que sean irreversibles, incluso para los administradores.')
    print()
    print('── TECNOLOGÍAS USADAS ───────────────────────────')
    print('• Python 3 + librerías: requests, google-generativeai, csv')
    print('• Cloud API: OpenWeatherMap (clima en tiempo real)')
    print('• IA Generativa: Google Gemini latest Flash')
    print('• Almacenamiento: archivos CSV locales')
    print()
    print('── EQUIPO ───────────────────────────────────────')
    print('Nombre del proyecto: GuardianClimaITBA')
    print('Desarrolladores:')
    print('  - Pedro Martinez')
    print('  - Francisco Martinez Freire')
    print('  - Pedro Miotti')
    print('  - Leandro Agustin Fernandez')
    print('─' * 50)


# ─────────────────────────────────────────
#  MENÚS PRINCIPALES
# ─────────────────────────────────────────

def menu_principal(username):
    """Menú principal post-login. Permanece activo hasta que el usuario cierra sesión."""
    ultimo_clima = None  # Guarda los datos del último clima consultado

    while True:
        limpiar_pantalla()
        print('=' * 45)
        print(f'  🌦  GUARDIÁNCLIMA ITBA  |  Hola, {username}!')
        print('=' * 45)
        print('  1. Consultar clima actual y guardar')
        print('  2. Ver mi historial personal por ciudad')
        print('  3. Estadísticas globales de uso')
        print('  4. Consejo IA: ¿cómo me visto hoy?')
        print('  5. Acerca de...')
        print('  6. Cerrar sesión')
        print('=' * 45)

        opcion = input('Elegí una opción (1-6): ').strip()

        if opcion == '1':
            resultado = consultar_clima_y_guardar(username)
            if resultado:
                ultimo_clima = resultado
        elif opcion == '2':
            ver_historial_personal(username)
        elif opcion == '3':
            mostrar_estadisticas_globales()
        elif opcion == '4':
            obtener_consejo_ia(ultimo_clima)
        elif opcion == '5':
            mostrar_acerca_de()
        elif opcion == '6':
            print(f'\nCerrando sesión de {username}... ¡Hasta pronto!')
            break
        else:
            print('⚠ Opción no válida. Ingresá un número del 1 al 6.')

        input('\nPresioná Enter para continuar...')


def menu_acceso():
    """Menú de acceso inicial (login / registro / salir)."""
    while True:
        limpiar_pantalla()
        print('=' * 45)
        print('     🌦  GUARDIÁNCLIMA ITBA  — BIENVENIDO')
        print('=' * 45)
        print('  1. Iniciar sesión')
        print('  2. Registrar nuevo usuario')
        print('  3. Salir')
        print('=' * 45)

        opcion = input('Elegí una opción (1-3): ').strip()

        if opcion == '1':
            username = iniciar_sesion()
            if username:
                menu_principal(username)
        elif opcion == '2':
            username = registrar_nuevo_usuario()
            if username:
                menu_principal(username)
        elif opcion == '3':
            print('\n¡Gracias por usar GuardiánClima ITBA! 👋')
            break
        else:
            print('⚠ Opción no válida.')
            input('Presioná Enter para continuar...')


# ─────────────────────────────────────────
#  PUNTO DE ENTRADA
# ─────────────────────────────────────────

inicializar_archivos()   # Crea los CSV si no existen
menu_acceso()            # Lanza la aplicación

---
## 📥 Descargar los archivos CSV generados

Una vez que uses la app, podés descargar los archivos ejecutando la celda de abajo,
o directamente desde el panel de archivos de Colab (ícono 📁 en la barra lateral).

In [ ]:
# ============================================================
# CELDA 6 — Descargar CSV generados (ejecutá cuando quieras)
# ============================================================
from google.colab import files

import os

if os.path.exists('historial_global.csv'):
    files.download('historial_global.csv')
    print('📥 historial_global.csv descargado.')
else:
    print('⚠ historial_global.csv no existe todavía (realizá al menos una consulta de clima).')

if os.path.exists('usuarios_simulados.csv'):
    files.download('usuarios_simulados.csv')
    print('📥 usuarios_simulados.csv descargado.')